# 3 — Model training (final, 50-minute plan) → `artifacts/models/`

**Input** (all in `artifacts/`): `pair_features_train.parquet` (file 2), `train_val_split.csv`, `ground_truth_clean.csv`,
`train_s1/s2/s3_clean.csv` (file 0)
**Output** (all in `artifacts/models/`): `l1_lightgbm.joblib`, `l1_catboost.joblib` (3 fold models each), `graph_r1.joblib`,
`graph_r2.joblib`, `best_model.json` (pipeline + decision rule for `4_predictions_final.py`), plus `models_final.py`
next to this notebook and `artifacts/feature_selection.csv`.
**Nothing old is used:** `models_final.py` is written by the first cell and imported from this folder (checked), every
model is trained fresh (`REFIT = True`), and the previous `l1_* / graph_* / best_model.json` are moved to
`artifacts/models/_previous/` at the start (`blocking_final_ranker.joblib` is never touched).

| Section | What it does | Minutes (plan) |
|---|---|---|
| **0. Setup** | writes + imports `models_final.py`, config, ids, the exact metric, loss breakdown, **50-min clock** | ~0.5 |
| **1. Quick check** | one fast LightGBM on 15% of the training S1s → F0.5 on the full validation fold | ~1.5 |
| **2. Feature selection** | thresholds only (empty/constant, near-duplicates, near-zero gain), capped at 25%, verified | ~1 |
| **3. Level-1, 3-fold** | **LightGBM** (16 min) + **CatBoost** (5 min): out-of-fold probability for every training pair, fold mean on validation | ~28 |
| **4. Graph re-score × 2** | group evidence: S1 side, candidate side, similarity to the S1's confident matches, **duplicate S1s** (shared candidates, same name/pin/house, word overlap of the two S1s) | ~13 |
| **5. Rule + leaderboard** | threshold, soft one-S1-per-candidate, per-rank thresholds (4 ranks), per country → `best_model.json`; loss breakdown | ~3 |

**Time guard:** `TOTAL_BUDGET_MIN = 50`. Every model stops by the clock, and each stage's budget shrinks automatically
if earlier stages ran long (CatBoost / round 2 are skipped rather than run over time). The last cell prints the time
used per stage.

**Score = the leaderboard metric:** macro F0.5 over **all** validation S1s (singletons and blocking misses included).

## 0. Setup

In [13]:
%%writefile models_final.py
"""
models_final.py — shared code of 3_model_training_final.ipynb (training) and 4_predictions_final.py (test).
Written by the notebook's first cell — edit it there.  Both files import THIS module, so the
level-2 features and the decision rule are computed by exactly the same code on validation and test.

  * time-budget callbacks (LightGBM / XGBoost / CatBoost stop by the clock)
  * model_proba / kfold_proba   probability of one fold model / the mean of a K-fold bundle
  * load_fields                 integer codes of a few clean-CSV columns (names, address parts)
                                + S1 word sets (sparse) for S1-vs-S1 duplicate similarity
  * level2_matrix               graph features of a probability vector: S1 side, candidate side,
                                similarity to the S1's confident matches, duplicate-S1 evidence
  * apply_rule                  decision rule: threshold + soft one-S1-per-candidate + per-rank
                                thresholds, optionally per country (France etc. -> default rule)
"""
import time

import numpy as np
import pandas as pd

PRED_CHUNK = 500_000

# clean-CSV columns used by the graph features (whatever is missing is simply skipped)
CAND_FIELDS = ["name_compact", "name_skeleton", "name_first_token", "address_core",
               "address_pin", "house_number", "street_name"]
S1_FIELDS = ["name_compact", "name_skeleton", "address_core", "address_pin", "house_number"]
S1_TOKENS = {"name": "name_tokens_sorted", "addr": "address_tokens_sorted"}   # word sets for S1-S1 Jaccard


# ============================================================================
#  time budgets
# ============================================================================
def lgb_time_limit(seconds):
    """LightGBM callback: stop after `seconds`, keeping the best iteration seen so far."""
    import lightgbm as lgb
    st = {"t0": None, "best": np.inf, "it": 0, "res": None}

    def _cb(env):
        if st["t0"] is None:
            st["t0"] = time.time()
        if env.evaluation_result_list:
            r = env.evaluation_result_list[0]
            v = -r[2] if r[3] else r[2]
            if v < st["best"]:
                st["best"], st["it"], st["res"] = v, env.iteration, env.evaluation_result_list
        if time.time() - st["t0"] > seconds:
            raise lgb.callback.EarlyStopException(st["it"], st["res"] or env.evaluation_result_list)
    _cb.order = 40
    return _cb


def xgb_time_limit(seconds):
    import xgboost as xgb

    class _T(xgb.callback.TrainingCallback):
        def __init__(self):
            super().__init__()
            self.t0 = None

        def after_iteration(self, model, epoch, evals_log):
            if self.t0 is None:
                self.t0 = time.time()
            return time.time() - self.t0 > seconds
    return _T()


class CatTimeLimit:
    def __init__(self, seconds):
        self.seconds, self.t0 = seconds, None

    def after_iteration(self, info):
        if self.t0 is None:
            self.t0 = time.time()
        return time.time() - self.t0 < self.seconds      # True = keep going


# ============================================================================
#  probabilities
# ============================================================================
def model_proba(m, X):
    """m = {"lib": lightgbm|xgboost|catboost|sklearn, "model": ..., "iters": n, "fill": v}"""
    lib, mod = m["lib"], m["model"]
    out = np.empty(len(X), np.float32)
    for lo in range(0, len(X), PRED_CHUNK):
        xb = np.asarray(X[lo:lo + PRED_CHUNK], dtype=np.float32)
        if m.get("fill") is not None:
            xb = np.where(np.isnan(xb), np.float32(m["fill"]), xb)
        if lib == "lightgbm":
            out[lo:lo + len(xb)] = mod.predict(xb, num_iteration=m.get("iters") or None)
        elif lib == "xgboost":
            import xgboost as xgb
            it = m.get("iters")
            out[lo:lo + len(xb)] = mod.predict(xgb.DMatrix(xb, missing=np.nan),
                                               iteration_range=(0, it) if it else (0, 0))
        else:
            out[lo:lo + len(xb)] = mod.predict_proba(xb)[:, 1]
    return out


def rows_proba(m, X, rows):
    """model_proba on X[rows] without copying all of X[rows] at once."""
    out = np.empty(len(rows), np.float32)
    for lo in range(0, len(rows), PRED_CHUNK):
        out[lo:lo + PRED_CHUNK] = model_proba(m, X[rows[lo:lo + PRED_CHUNK]])
    return out


def kfold_proba(bundle, X):
    """test / validation probability of a K-fold bundle = mean of its fold models."""
    return np.mean([model_proba(m, X) for m in bundle["models"]], axis=0).astype(np.float32)


# ============================================================================
#  group helpers
# ============================================================================
def group_max(v, g, n=None):
    n = int(g.max()) + 1 if n is None else n
    m = np.full(n, -np.inf)
    np.maximum.at(m, g, v)
    return m[g]


def rank_lead(g, v):
    """rank of each row inside its group (0 = highest v) and v minus the best OTHER value
    of the group (NaN for a group of one)."""
    n = len(v)
    vf = np.where(np.isnan(v), -np.inf, v).astype(np.float64)
    o = np.lexsort((-vf, g))
    go = g[o]
    st = np.flatnonzero(np.concatenate(([True], go[1:] != go[:-1])))
    gl = np.diff(np.append(st, n))
    first = np.repeat(st, gl)
    rank = np.empty(n, np.float32)
    rank[o] = np.arange(n) - first
    vo = vf[o]
    best = np.repeat(vo[st], gl)
    sec = np.repeat(np.where(gl >= 2, vo[np.minimum(st + 1, n - 1)], -np.inf), gl)
    other = np.empty(n, np.float64)
    other[o] = np.where(np.arange(n) == first, sec, best)
    with np.errstate(invalid="ignore"):
        lead = (vf - other).astype(np.float32)
    lead[~np.isfinite(other) | ~np.isfinite(vf)] = np.nan
    return rank, lead


def _top2_other(g, v, s):
    """for every row: the `s` value of the best OTHER row of its group `g` (by v), -1 if none."""
    n = len(v)
    o = np.lexsort((-v, g))
    go = g[o]
    st = np.flatnonzero(np.concatenate(([True], go[1:] != go[:-1])))
    gl = np.diff(np.append(st, n))
    so = s[o]
    t1 = so[st]
    t2 = np.where(gl >= 2, so[np.minimum(st + 1, n - 1)], -1)
    first = np.repeat(st, gl)
    is_first = np.empty(n, bool)
    is_first[o] = np.arange(n) == first
    gid = np.empty(n, np.int64)
    gid[o] = np.repeat(np.arange(len(st)), gl)
    return np.where(is_first, t2[gid], t1[gid])


def _lookup(keys_sorted, vals, q):
    """vals[k] where keys_sorted[k] == q, else 0."""
    if len(keys_sorted) == 0:
        return np.zeros(len(q), np.float64)
    pos = np.minimum(np.searchsorted(keys_sorted, q), len(keys_sorted) - 1)
    return np.where(keys_sorted[pos] == q, vals[pos], 0).astype(np.float64)


# ============================================================================
#  clean-CSV field codes
# ============================================================================
def load_fields(art, split, s1_cats, cand_cats):
    """-> (s1 fields, candidate fields): {column: int32 code per category}, -1 = empty.
    Codes are only compared within the same table side, so each column is factorised alone."""
    from pathlib import Path
    art = Path(art)

    def read(name, fields):
        f = art / f"{split}_{name}_clean.csv"
        if not f.exists():
            return None
        have = pd.read_csv(f, nrows=0).columns
        use = ["entity_id"] + [c for c in fields if c in have]
        return pd.read_csv(f, usecols=use, dtype=str, keep_default_na=False)

    def codes(df, cats, fields):
        out = {}
        if df is None:
            return out
        df = df.drop_duplicates("entity_id").set_index("entity_id")
        for c in fields:
            if c not in df:
                continue
            v = df[c].reindex(pd.Index(cats)).fillna("").astype(str).str.strip().to_numpy(dtype=object)
            k, _ = pd.factorize(v)
            k = k.astype(np.int32)
            k[v == ""] = -1
            out[c] = k
        return out

    s1 = read("s1", S1_FIELDS + list(S1_TOKENS.values()))
    cd = [d for d in (read("s2", CAND_FIELDS), read("s3", CAND_FIELDS)) if d is not None]
    cd = pd.concat(cd, ignore_index=True) if cd else None
    sf = codes(s1, s1_cats, S1_FIELDS)
    if s1 is not None:
        s1i = s1.drop_duplicates("entity_id").set_index("entity_id")
        for key, colname in S1_TOKENS.items():
            if colname in s1i:
                m = _tokens_csr(s1i[colname].reindex(pd.Index(s1_cats)).fillna("").astype(str).to_numpy(dtype=object))
                if m is not None:
                    sf[f"_tok_{key}"] = m
    return sf, codes(cd, cand_cats, CAND_FIELDS)


def _tokens_csr(values):
    """one row per value, a 1 for every distinct word (scipy CSR); None without scipy."""
    try:
        from scipy.sparse import csr_matrix
    except Exception:
        return None
    t = pd.Series(values, index=np.arange(len(values))).str.split().explode()
    t = t[t.notna() & (t != "")]
    n = len(values)
    if not len(t):
        return csr_matrix((n, 1), dtype=np.float32)
    code, _ = pd.factorize(t.to_numpy())
    m = csr_matrix((np.ones(len(code), np.float32), (t.index.to_numpy(), code)), shape=(n, int(code.max()) + 1))
    m.sum_duplicates()
    m.data[:] = 1.0
    return m


def _pair_jaccard(M, a, b, chunk=2_000_000):
    """Jaccard of the word sets of rows a[i] and b[i] of M (NaN when both are empty)."""
    ns = M.shape[0]
    key = a.astype(np.int64) * ns + b.astype(np.int64)
    uk, inv = np.unique(key, return_inverse=True)
    out = np.full(len(uk), np.nan)
    for lo in range(0, len(uk), chunk):
        k = uk[lo:lo + chunk]
        A, B = M[k // ns], M[k % ns]
        inter = np.asarray(A.multiply(B).sum(axis=1)).ravel()
        uni = np.diff(A.indptr) + np.diff(B.indptr) - inter
        out[lo:lo + chunk] = np.where(uni > 0, inter / np.maximum(uni, 1), np.nan)
    return out[inv]


# ============================================================================
#  level-2 (graph) features
# ============================================================================
def level2_names(extra_names, cf_fields, sf_fields, raw_names, tok_keys=()):
    n = ["p"] + [f"x_{e}" for e in extra_names] + (["x_std"] if len(extra_names) >= 2 else [])
    n += ["s1_rank", "s1_lead", "p_div_s1max", "s1_sum", "s1_n50", "s1_n90", "s1_npairs", "rank_minus_n50"]
    n += ["c_rank", "c_lead", "p_div_cmax", "c_sum", "c_n50", "c_n90", "c_npairs"]
    n += ["anchor_n_other"] + [f"anchor_share_{f}" for f in cf_fields] + ["anchor_share_max"]
    n += ["other_s1_p", "other_s1_shared", "other_s1_shared_frac", "s1_dup_shared_max"]
    n += [f"other_s1_same_{f}" for f in sf_fields] + [f"other_s1_jacc_{t}" for t in tok_keys]
    n += [f"raw_{r}" for r in raw_names]
    return n


def level2_matrix(p, extras, raw, raw_names, s1, cand, sf, cf, anchor_p=0.9, conf_p=0.5, max_share=50):
    """p: main probability (all rows of the split); extras: list of (name, prob vector);
    raw: (n, k) raw pair features; s1/cand: integer codes; sf/cf: field codes per S1 / candidate code.
    Returns (Z float32 (n, d), names).  Everything is computed over ALL rows given — candidates
    compete across every S1 they appear with (validation and test alike)."""
    p = np.asarray(p, np.float64)
    n = len(p)
    cff = sorted(cf)
    sff = sorted(k for k in sf if not k.startswith("_"))
    toks = sorted(k[5:] for k in sf if k.startswith("_tok_"))
    names = level2_names([e for e, _ in extras], cff, sff, raw_names, toks)
    Z = np.full((n, len(names)), np.nan, np.float32)
    col = {c: j for j, c in enumerate(names)}

    def put(name, v):
        Z[:, col[name]] = v

    put("p", p)
    if extras:
        E = np.column_stack([np.asarray(v, np.float32) for _, v in extras])
        for j, (e, _) in enumerate(extras):
            put(f"x_{e}", E[:, j])
        if len(extras) >= 2:
            put("x_std", E.std(axis=1))
        del E

    # ---- S1 side ----
    ns = int(s1.max()) + 1
    r1, l1 = rank_lead(s1, p)
    mx = group_max(p, s1, ns)
    n50 = np.bincount(s1, weights=(p >= 0.5), minlength=ns)
    n90 = np.bincount(s1, weights=(p >= anchor_p), minlength=ns)
    put("s1_rank", r1); put("s1_lead", l1); put("p_div_s1max", p / np.maximum(mx, 1e-6))
    put("s1_sum", np.bincount(s1, weights=p, minlength=ns)[s1]); put("s1_n50", n50[s1]); put("s1_n90", n90[s1])
    put("s1_npairs", np.bincount(s1, minlength=ns)[s1]); put("rank_minus_n50", r1 - n50[s1])
    del mx

    # ---- candidate side ----
    nc = int(cand.max()) + 1
    rc, lc = rank_lead(cand, p)
    cmx = group_max(p, cand, nc)
    put("c_rank", rc); put("c_lead", lc); put("p_div_cmax", p / np.maximum(cmx, 1e-6))
    put("c_sum", np.bincount(cand, weights=p, minlength=nc)[cand])
    put("c_n50", np.bincount(cand, weights=(p >= 0.5), minlength=nc)[cand])
    put("c_n90", np.bincount(cand, weights=(p >= anchor_p), minlength=nc)[cand])
    put("c_npairs", np.bincount(cand, minlength=nc)[cand])
    put("other_s1_p", p - lc)                       # best OTHER S1's p for this candidate (NaN: none)
    del rc, lc, cmx

    # ---- similarity to the S1's confident matches (anchors) ----
    is_a = p >= anchor_p
    n_other = n90[s1] - is_a
    put("anchor_n_other", n_other)
    shares = []
    for f in cff:
        code = cf[f][cand].astype(np.int64)
        ok = code >= 0
        key = s1.astype(np.int64) * (int(code.max()) + 2) + code
        uk, cnt = np.unique(key[is_a & ok], return_counts=True)
        c = _lookup(uk, cnt, key) - (is_a & ok)
        sh = np.where(ok & (n_other > 0), c / np.maximum(n_other, 1), np.nan)
        put(f"anchor_share_{f}", sh)
        shares.append(sh)
    if shares:
        S = np.column_stack(shares)
        allnan = np.isnan(S).all(axis=1)
        put("anchor_share_max", np.where(allnan, np.nan, np.nanmax(np.where(np.isnan(S), -1, S), axis=1)))
        del S

    # ---- the candidate's best OTHER S1: how strong, and is it a duplicate of this S1? ----
    other = _top2_other(cand, p, s1.astype(np.int64))
    has_o = other >= 0
    o_safe = np.where(has_o, other, 0)
    conf = p >= conf_p
    cc = cand[conf]
    cnt = np.bincount(cc, minlength=nc)
    keep = (cnt[cc] >= 2) & (cnt[cc] <= max_share)
    shared = np.zeros(n)
    dupmax = np.zeros(ns)
    if keep.any():
        d = pd.DataFrame({"s": s1[conf][keep].astype(np.int64), "c": cc[keep]})
        m = d.merge(d, on="c")
        m = m[m["s_x"] != m["s_y"]]
        if len(m):
            uk, kc = np.unique(m["s_x"].to_numpy() * ns + m["s_y"].to_numpy(), return_counts=True)
            shared = _lookup(uk, kc, s1.astype(np.int64) * ns + o_safe)
            np.maximum.at(dupmax, uk // ns, kc)            # most likely candidates shared with ANY other S1
        del d, m
    put("other_s1_shared", np.where(has_o, shared, np.nan))
    put("other_s1_shared_frac", np.where(has_o, shared / np.maximum(n50[s1], 1), np.nan))
    put("s1_dup_shared_max", dupmax[s1])
    for f in sff:
        a, b = sf[f][s1], sf[f][o_safe]
        put(f"other_s1_same_{f}", np.where(has_o & (a >= 0) & (b >= 0), (a == b).astype(np.float32), np.nan))
    for t in toks:                                          # word overlap of this S1 and the other S1
        v = np.full(n, np.nan)
        if has_o.any():
            idx = np.flatnonzero(has_o)
            v[idx] = _pair_jaccard(sf[f"_tok_{t}"], s1[idx], other[idx])
        put(f"other_s1_jacc_{t}", v)

    # ---- raw pair features ----
    for j, r in enumerate(raw_names):
        put(f"raw_{r}", raw[:, j])
    return Z, names


# ============================================================================
#  decision rule
# ============================================================================
def _decide(p, s1, cand, r, cmax):
    """match if p >= thr and p >= opc_r * (the candidate's best p over all its S1s)
    [opc_r = 1: a candidate goes only to its best S1; < 1: also to S1s almost as good; 0: off],
    OR the pair is ranked k-th among its S1's surviving pairs and p >= thr_ranks[k]."""
    base = p >= float(r.get("opc_r", 1.0)) * cmax - 1e-12
    pred = base & (p >= float(r["thr"]))
    ranks = [(int(k), float(t)) for k, t in (r.get("thr_ranks") or []) if float(t) < float(r["thr"])]
    if ranks:
        rank, _ = rank_lead(s1, np.where(base, p, -np.inf))
        for k, t in ranks:
            pred |= base & (rank == k) & (p >= t)
    return pred


def apply_rule(p, s1, cand, rule, country=None, cmax=None):
    """rule = {"default": {thr, opc_r, thr_ranks}, "by_country": {country: {...}}}.
    cmax = each row's candidate-best p (computed over all rows given when None)."""
    p = np.asarray(p, np.float64)
    if cmax is None:
        cmax = group_max(p, cand)
    by = rule.get("by_country") or {}
    if country is None or not by:
        return _decide(p, s1, cand, rule["default"], cmax)
    pred = np.zeros(len(p), bool)
    country = np.asarray(country, dtype=object)
    for c in pd.unique(country):
        m = country == c
        pred[m] = _decide(p[m], s1[m], cand[m], by.get(c, rule["default"]), cmax[m])
    return pred


def round_inputs(rnd, l1, prev=None):
    """inputs of graph round `rnd` (1-based): (main probability, extra probability columns).
    l1 = {model name: probability} in a fixed order; prev = the previous round's probability."""
    names = list(l1)
    p1 = np.mean([np.asarray(l1[m], np.float64) for m in names], axis=0)
    ex = [(m, l1[m]) for m in names] if len(names) >= 2 else []
    if rnd == 1:
        return p1, ex
    return prev, ex + [("l1_mean" if len(names) >= 2 else "l1", p1)]

Overwriting models_final.py


In [14]:
# ---------- config ----------
import gc, json, os, sys, time, warnings, importlib
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import joblib, shutil
warnings.filterwarnings("ignore")
T_RUN0 = time.time()                    # the 50-minute clock starts here

BASE      = Path.cwd()                  # the folder with artifacts/ (same folder as 1_blocking_final.py)
ART       = BASE / "artifacts"
MODEL_DIR = ART / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(BASE))
for _m in [m for m in list(sys.modules) if m.split(".")[-1] == "models_final"]:
    del sys.modules[_m]                 # always the models_final.py written by the cell above
import models_final as MF
assert Path(MF.__file__).resolve() == (BASE / "models_final.py").resolve(), \
    f"imported {MF.__file__} instead of {BASE / 'models_final.py'} — run this notebook from the dataset folder"
print("models_final:", MF.__file__)

SEED      = 42
ID_COLS   = ["source1_entity_id", "candidate_entity_id"]
META_COLS = ["country", "fold"]         # never features (France never appears in training)
TARGET    = "label"

# ---- level-1 models (K-fold by S1) ----
TOTAL_BUDGET_MIN = 50                   # the whole notebook, start to finish
L1_MODELS        = ["lightgbm", "catboost"]            # also possible: "xgboost"
L1_BUDGET_MIN    = {"lightgbm": 16, "catboost": 5, "xgboost": 6}   # training minutes per model, all folds together
K_FOLDS          = 3
L1_LR            = 0.1
ROW_CAP          = {"xgboost": 3_000_000, "catboost": 2_500_000}   # fit rows per fold (slower libraries)
ES_FRAC          = 0.05                 # of each fold's training S1s held out for early stopping
REFIT            = True                 # True: train everything fresh.  False only to resume after a crash

# ---- level-2 graph re-score ----
GRAPH_ROUNDS     = 2                    # 0 = off   (a 3rd round added nothing: 0.9898 -> 0.9898)
ROUND_BUDGET_MIN = 5                    # training minutes per round, all K folds together
L2_LR            = 0.08
N_RAW            = 20                   # highest-gain pair features passed to level 2
ANCHOR_P         = 0.90                 # a pair this sure is one of its S1's "confident matches"
CONF_P           = 0.50                 # "likely" (duplicate-S1 evidence)

# ---- decision rule ----
THR_GRID      = np.round(np.arange(0.05, 0.99, 0.01), 2)     # per-rank thresholds search the whole range
THR_MAIN      = np.round(np.arange(0.30, 0.96, 0.01), 2)     # the main threshold (0.67-0.71 in every run so far)
OPC_GRID      = (1.0, 0.97, 0.9, 0.0)   # 1 = a candidate goes only to its best S1; <1 soft; 0 off
RANK_MAX      = 4                       # extra thresholds for each S1's 1st..4th surviving pair (recall)
MIN_PRECISION = 0.996                   # rules below this validation precision are avoided ...
PREC_SLACK    = 0.0003                  # ... unless that costs more than this much F0.5

# ---- quick check / feature selection (thresholds, never a hand-written list) ----
QUICK_S1_FRAC = 0.15
STAT_SAMPLE   = 300_000
DEAD_COVERAGE = 0.0005
DUP_CORR      = 0.99
DUP_OVERLAP   = 0.90
GAIN_MIN      = 1e-4
PROTECT_AUC   = 0.20
MAX_DROP_FRAC = 0.25
DROP_TOL      = 0.0002
N_JOBS, READ_COLS = -1, 16

def have(m):
    try:
        __import__(m); return True
    except Exception:
        return False
LIBS = {m: have(m) for m in ["lightgbm", "xgboost", "catboost", "scipy"]}
print("libraries:", LIBS)
assert LIBS["lightgbm"], "pip install lightgbm"
import lightgbm as lgb
L1_USED = [m for m in L1_MODELS if m == "lightgbm" or LIBS.get(m)]
print("level-1 models:", L1_USED)

# ---- the clock ----
TAIL_MIN, FIELDS_MIN, ROUND_OVERHEAD = 3.5, 2.0, 1.3   # minutes: rule tuning + reports; fields; per-round scoring
STAGE_MIN = {}
def used_min():
    return (time.time() - T_RUN0) / 60
def left_min():
    return TOTAL_BUDGET_MIN - used_min()
def l1_overhead(name, b):
    """scoring + data copies on top of a model's training minutes (measured on the full data)."""
    return 0.2 * b + 0.8 if name == "lightgbm" else 1.5

# ---- nothing old: move the previous run's models aside (the blocking ranker stays) ----
if REFIT:
    olds = [f for f in MODEL_DIR.iterdir() if f.is_file()
            and (f.name.startswith(("l1_", "graph_r")) or f.name == "best_model.json")]
    if olds:
        (MODEL_DIR / "_previous").mkdir(exist_ok=True)
        for f in olds:
            shutil.move(str(f), str(MODEL_DIR / "_previous" / f.name))
        print(f"moved {len(olds)} previous model files -> artifacts/models/_previous/")
print("blocking ranker present:", (MODEL_DIR / "blocking_final_ranker.joblib").exists(), "(needed by file 4, not here)")

models_final: /Users/mohnish/Desktop/AMAZON/student_resource/dataset/models_final.py
libraries: {'lightgbm': True, 'xgboost': True, 'catboost': True, 'scipy': True}
level-1 models: ['lightgbm', 'catboost']
moved 9 previous model files -> artifacts/models/_previous/
blocking ranker present: True (needed by file 4, not here)


In [15]:
# ---------- ids, labels, countries, splits, S1 folds (integer codes only) ----------
t0 = time.time()
PATH = ART / "pair_features_train.parquet"
pf = pq.ParquetFile(PATH)
cols = pf.schema_arrow.names
ALL_FEATURES = [c for c in cols if c not in ID_COLS + META_COLS + [TARGET]]
ids = pf.read(columns=ID_COLS + [TARGET] + (["country"] if "country" in cols else [])).to_pandas(strings_to_categorical=True)
S1_CATS = pd.Index(ids["source1_entity_id"].cat.categories)
CD_CATS = pd.Index(ids["candidate_entity_id"].cat.categories)
s1_raw = ids["source1_entity_id"].cat.codes.to_numpy().astype(np.int64)
cd_raw = ids["candidate_entity_id"].cat.codes.to_numpy().astype(np.int64)
y_raw  = ids[TARGET].to_numpy(dtype=np.int8)
if "country" in ids:
    ctry_raw = np.asarray(ids["country"].cat.categories, dtype=object)[ids["country"].cat.codes.to_numpy()]
else:
    ctry_raw = np.full(len(ids), "all", dtype=object)
del ids

split = pd.read_csv(ART / "train_val_split.csv", dtype=str)
val_s1 = split.loc[split["split"] == "val", "entity_id"].to_numpy(dtype=object)

def s1_flag(idlist):
    f = np.zeros(len(S1_CATS), dtype=bool)
    i = S1_CATS.get_indexer(idlist)
    f[i[i >= 0]] = True
    return f

F_VAL = s1_flag(val_s1)
ROWS_VAL = np.flatnonzero(F_VAL[s1_raw])
ROWS_TR  = np.flatnonzero(~F_VAL[s1_raw])

rng = np.random.default_rng(SEED)
tr_codes = np.unique(s1_raw[ROWS_TR])
perm = rng.permutation(len(tr_codes))
S1_FOLD = np.full(len(S1_CATS), -1, np.int8)            # fold of every training S1, -1 = validation
S1_FOLD[tr_codes[perm]] = np.arange(len(perm)) % K_FOLDS
nq = max(1, int(len(perm) * QUICK_S1_FRAC)); ne = max(1, int(len(perm) * ES_FRAC))
F_QUICK = np.zeros(len(S1_CATS), bool); F_QUICK[tr_codes[perm[:nq]]] = True
F_QES   = np.zeros(len(S1_CATS), bool); F_QES[tr_codes[perm[nq:nq + ne]]] = True
ROWS_QF = np.flatnonzero(F_QUICK[s1_raw])
ROWS_QE = np.flatnonzero(F_QES[s1_raw])

s1_country = pd.read_csv(ART / "train_s1_clean.csv", usecols=["entity_id", "country_clean"], dtype=str,
                         keep_default_na=False).drop_duplicates("entity_id").set_index("entity_id")["country_clean"].str.strip()
print(f"{len(y_raw):,} pairs x {len(ALL_FEATURES)} features | train {len(ROWS_TR):,} ({len(tr_codes):,} S1, {K_FOLDS} folds)"
      f" | val {len(ROWS_VAL):,} | quick {len(ROWS_QF):,}   positive rate {y_raw.mean():.3f}   [{time.time()-t0:.1f}s]")
print("countries (pairs):", pd.Series(ctry_raw).value_counts().to_dict())

def load_rows(rows, feats):
    """parquet -> float32 matrix of the given rows / columns, a few columns at a time."""
    X = np.empty((len(rows), len(feats)), dtype=np.float32)
    for k in range(0, len(feats), READ_COLS):
        part = feats[k:k + READ_COLS]
        tb = pf.read(columns=part)
        for j, cname in enumerate(part):
            X[:, k + j] = np.asarray(tb.column(cname).to_numpy(), dtype=np.float32)[rows]
        del tb
    return X

13,550,327 pairs x 158 features | train 11,518,129 (1,857,750 S1, 3 folds) | val 2,032,198 | quick 1,728,465   positive rate 0.561   [9.8s]
countries (pairs): {'US': 7544680, 'India': 6005647}


In [16]:
# ---------- the competition metric: macro F0.5 over ALL S1s of a fold, + where it is lost ----------
gt = pd.read_csv(ART / "ground_truth_clean.csv", dtype=str, keep_default_na=False)
_ids = gt["matched_entity_ids"].str.strip()
N_TRUE = pd.Series(np.where(_ids == "", 0, _ids.str.count(",") + 1), index=gt["source1_entity_id"].to_numpy())
N_TRUE = N_TRUE[~N_TRUE.index.duplicated()]
del gt, _ids

class F05Evaluator:
    def __init__(self, universe_ids, s1c, cdc, y):
        uni = pd.unique(np.asarray(universe_ids, dtype=object))
        nt = N_TRUE.reindex(uni).fillna(0).to_numpy(dtype=np.float64)
        code = S1_CATS.get_indexer(uni)
        has = code >= 0
        loc = np.full(len(S1_CATS), -1, dtype=np.int64)
        loc[code[has]] = np.arange(int(has.sum()))
        self.s1 = loc[s1c]
        assert (self.s1 >= 0).all(), "a pair's S1 is outside the evaluation universe"
        self.cand = cdc
        self.y = y.astype(bool)
        self.n = int(has.sum())
        self.n_true = nt[has]
        self.n_univ = len(uni)
        self.x_single = int(((~has) & (nt == 0)).sum())          # singletons without candidates: always right
        self.x_miss = int(((~has) & (nt > 0)).sum())             # S1s whose matches blocking lost completely
        self.n_single = int((self.n_true == 0).sum()) + self.x_single
        self.n_matched = self.n_univ - self.n_single
        self.total_true = nt.sum()

    def per_s1(self, pred):
        npred = np.bincount(self.s1[pred], minlength=self.n).astype(np.float64)
        tp = np.bincount(self.s1[pred & self.y], minlength=self.n).astype(np.float64)
        with np.errstate(divide="ignore", invalid="ignore"):
            prec = np.where(npred > 0, tp / npred, 0.0)
            rec = np.where(self.n_true > 0, tp / self.n_true, 0.0)
            f = np.where(tp > 0, 1.25 * prec * rec / (0.25 * prec + rec), 0.0)
        single = self.n_true == 0
        f = np.where(single, (npred == 0).astype(float), f)
        return f, npred, tp, single

    def score(self, pred):
        f, npred, tp, single = self.per_s1(pred)
        return dict(f05=(f.sum() + self.x_single) / self.n_univ,
                    precision=tp.sum() / max(npred.sum(), 1), recall=tp.sum() / max(self.total_true, 1),
                    f05_matched=f[~single].sum() / max(self.n_matched, 1),
                    singleton_acc=(f[single].sum() + self.x_single) / max(self.n_single, 1),
                    n_pred=int(npred.sum()))

    def sweep(self, p):
        """quick report: best (hard one-per-candidate on/off, threshold) on this fold's rows only."""
        p = np.asarray(p, dtype=np.float64)
        cbest = p >= MF.group_max(p, self.cand)
        best = None
        for opc in (True, False):
            base = cbest if opc else np.ones(len(p), bool)
            for thr in THR_GRID:
                r = self.score(base & (p >= thr))
                if best is None or r["f05"] > best["f05"]:
                    best = dict(r, thr=float(thr), one_per_cand=opc)
        return best

    def loss_report(self, pred, title=""):
        """F0.5 points lost (sum = 1 - F0.5), by what went wrong and by how many true matches the S1 has."""
        f, npred, tp, single = self.per_s1(pred)
        ntc = np.bincount(self.s1[self.y], minlength=self.n).astype(np.float64)
        fp, fn_c, fn_b = npred - tp, ntc - tp, self.n_true - ntc
        kind = np.select([single & (npred > 0), (fp > 0) & (fn_c > 0), fp > 0, fn_c > 0, fn_b > 0],
                         ["singleton predicted", "wrong + missed", "wrong pair(s)", "missed pair(s)",
                          "blocking miss only"], "perfect")
        nt = self.n_true
        bucket = np.select([nt == 0, nt == 1, nt == 2, nt <= 5, nt <= 10], ["0", "1", "2", "3-5", "6-10"], "11+")
        d = pd.DataFrame(dict(kind=kind, bucket=bucket, loss=1.0 - f))
        t = d.pivot_table(index="bucket", columns="kind", values="loss", aggfunc="sum", fill_value=0.0)
        t = t.drop(columns=[c for c in ["perfect"] if c in t]) / self.n_univ
        t["S1s"] = d.groupby("bucket").size()
        t.loc["no candidates"] = 0.0
        t.loc["no candidates", "S1s"] = self.x_miss + self.x_single
        if "blocking miss only" not in t:
            t["blocking miss only"] = 0.0
        t.loc["no candidates", "blocking miss only"] = self.x_miss / self.n_univ
        t = t.reindex([b for b in ["0", "1", "2", "3-5", "6-10", "11+", "no candidates"] if b in t.index])
        t["total lost"] = t.drop(columns=["S1s"]).sum(axis=1)
        t.loc["ALL"] = t.sum()
        t = t[["S1s"] + [c for c in t.columns if c not in ("S1s", "total lost")] + ["total lost"]]
        print(f"\nF0.5 points lost {title}  (F0.5 = 1 - ALL.total lost)")
        print(t.to_string(formatters={"S1s": "{:,.0f}".format}, float_format=lambda v: f"{v:.5f}"))
        return t

EV_VAL = F05Evaluator(val_s1, s1_raw[ROWS_VAL], cd_raw[ROWS_VAL], y_raw[ROWS_VAL])
oracle = EV_VAL.score(EV_VAL.y.copy())
print(f"validation: {EV_VAL.n_univ:,} S1 ({EV_VAL.n_single / EV_VAL.n_univ:.1%} singletons), "
      f"{EV_VAL.total_true:,.0f} true pairs, blocking recall ceiling {EV_VAL.y.sum() / EV_VAL.total_true:.4f}")
print(f"oracle (perfect classifier on these candidates): F0.5 = {oracle['f05']:.4f}")

def show(name, r, secs=None):
    rule = r.get("rule")
    extra = (f"thr={rule['thr']:.2f} opc_r={rule['opc_r']} ranks={rule['thr_ranks']}" if rule else
             f"thr={r.get('thr', float('nan')):.2f} one_per_cand={r.get('one_per_cand')}")
    print(f"[{name}]  VALIDATION  F0.5 = {r['f05']:.4f}   precision = {r['precision']:.4f}   recall = {r['recall']:.4f}"
          f"   ({extra})" + (f"   [{secs:.0f}s]" if secs is not None else ""))

validation: 331,023 S1 (5.7% singletons), 1,144,441 true pairs, blocking recall ceiling 0.9952
oracle (perfect classifier on these candidates): F0.5 = 0.9985


## 1. Quick check (~3 min)
One LightGBM with fixed parameters on the quick S1s (15% of the training S1s), early-stopped, scored on the **full**
validation fold with the real metric.

In [17]:
t0 = time.time()
ROWS_Q = np.concatenate([ROWS_QF, ROWS_QE, ROWS_VAL])
XQ = load_rows(ROWS_Q, ALL_FEATURES)
YQ = y_raw[ROWS_Q]
QF, QE, QV = slice(0, len(ROWS_QF)), slice(len(ROWS_QF), len(ROWS_QF) + len(ROWS_QE)), slice(len(ROWS_QF) + len(ROWS_QE), None)
print(f"quick matrix {XQ.shape[0]:,} x {XQ.shape[1]}  ({XQ.nbytes / 1e9:.2f} GB)   [{time.time()-t0:.0f}s]")

QUICK_PARAMS = dict(objective="binary", learning_rate=0.1, num_leaves=127, min_child_samples=100,
                    subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                    n_jobs=N_JOBS, random_state=SEED, verbose=-1)

def quick_fit(feats):
    idx = [ALL_FEATURES.index(f) for f in feats]
    full = len(idx) == len(ALL_FEATURES)
    Xf, Xe, Xv = (XQ[QF], XQ[QE], XQ[QV]) if full else (XQ[QF][:, idx], XQ[QE][:, idx], XQ[QV][:, idx])
    est = lgb.LGBMClassifier(n_estimators=2000, **QUICK_PARAMS)
    est.fit(Xf, YQ[QF], eval_set=[(Xe, YQ[QE])], callbacks=[lgb.early_stopping(50, verbose=False)])
    p = est.predict_proba(Xv)[:, 1]
    return est, p, EV_VAL.sweep(p)

t = time.time()
QUICK, p_quick, r_quick = quick_fit(ALL_FEATURES)
show("quick_lgbm", r_quick, time.time() - t)
STAGE_MIN["setup + quick check"] = used_min()
print(f"          trees = {QUICK.best_iteration_}   features = {len(ALL_FEATURES)}   (oracle {oracle['f05']:.4f})")

quick matrix 4,335,438 x 158  (2.74 GB)   [47s]
[quick_lgbm]  VALIDATION  F0.5 = 0.9874   precision = 0.9970   recall = 0.9705   (thr=0.71 one_per_cand=True)   [44s]
          trees = 431   features = 158   (oracle 0.9985)


## 2. Feature selection — thresholds, capped, verified
1. **empty / constant** — present in < `DEAD_COVERAGE` of rows, or a single value
2. **near-duplicates** — |r| > `DUP_CORR` on rows where both exist, only when both exist on (almost) the same rows
3. **near-zero gain** — quick-model gain share < `GAIN_MIN`, **unless** strong on its own rows (|AUC−0.5| ≥ `PROTECT_AUC`)
4. at most `MAX_DROP_FRAC` of the features go; the quick model is refit — if F0.5 drops by more than `DROP_TOL`,
   the gain-based drops are reverted. Result → `artifacts/feature_selection.csv` (reused after a restart).

In [18]:
t0 = time.time()
k = len(ALL_FEATURES)
nqf = XQ[QF].shape[0]
SI = np.sort(rng.choice(nqf, size=min(STAT_SAMPLE, nqf), replace=False))
Xs, ys = XQ[QF][SI], YQ[QF][SI]
present = ~np.isnan(Xs)
cov = present.mean(axis=0)
lo_, hi_ = np.nanmin(np.where(present, Xs, np.inf), axis=0), np.nanmax(np.where(present, Xs, -np.inf), axis=0)

def auc_fast(y, v):
    ok = ~np.isnan(v); y, v = y[ok], v[ok]
    n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0 or len(v) < 50:
        return np.nan
    r = pd.Series(v).rank().to_numpy()
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

auc = np.array([auc_fast(ys, Xs[:, j]) for j in range(k)])
sep = np.nan_to_num(np.abs(auc - 0.5), nan=0.0)
gain = QUICK.booster_.feature_importance(importance_type="gain").astype(np.float64)
gshare = gain / max(gain.sum(), 1e-12)
REPORT = pd.DataFrame(dict(coverage=cov, auc=auc, sep=sep, gain_share=gshare), index=ALL_FEATURES)
REPORT["drop"] = ""

dead = [f for j, f in enumerate(ALL_FEATURES) if cov[j] < DEAD_COVERAGE or not (hi_[j] > lo_[j])]
REPORT.loc[dead, "drop"] = "empty/constant"
live = [j for j, f in enumerate(ALL_FEATURES) if f not in dead]
P_ = present[:, live].astype(np.float32)
both = P_.T @ P_
cnt = np.diag(both)
C = pd.DataFrame(Xs[:, live]).corr(min_periods=1000).abs().to_numpy()
iu = np.triu_indices(len(live), 1)
cv = np.nan_to_num(C[iu], nan=0.0)
ok = (cv > DUP_CORR) & (both[iu] >= DUP_OVERLAP * np.maximum(cnt[iu[0]], cnt[iu[1]]))
cand_pairs = sorted(zip(cv[ok], iu[0][ok], iu[1][ok]), reverse=True)
dup, gone = [], set(dead)
for r_, a_, b_ in cand_pairs:
    fa, fb = ALL_FEATURES[live[a_]], ALL_FEATURES[live[b_]]
    if fa in gone or fb in gone:
        continue
    key = lambda f: (REPORT.at[f, "gain_share"], REPORT.at[f, "sep"] * REPORT.at[f, "coverage"])
    lose, win = (fa, fb) if key(fa) < key(fb) else (fb, fa)
    dup.append((lose, win, r_)); gone.add(lose)
    REPORT.at[lose, "drop"] = f"duplicate of {win} (|r|={r_:.3f})"

low = REPORT[(REPORT["drop"] == "") & (REPORT["gain_share"] < GAIN_MIN) & (REPORT["sep"] < PROTECT_AUC)]
low = low.sort_values("gain_share").index.tolist()
cap = int(MAX_DROP_FRAC * k)
fixed = dead + [d[0] for d in dup]
if len(fixed) > cap:
    fixed = dead + [d[0] for d in dup][:max(0, cap - len(dead))]
low = low[:max(0, cap - len(fixed))]
REPORT.loc[low, "drop"] = "near-zero gain"
print(f"empty / constant ({len(dead)}): {dead}")
print(f"near-duplicates ({len(dup)}):")
for l_, w_, r_ in dup:
    print(f"    drop {l_:<28} keep {w_:<28} |r| = {r_:.4f}")
print(f"near-zero gain ({len(low)}): {low}")

KEEP = [f for f in ALL_FEATURES if f not in set(fixed) | set(low)]
if len(KEEP) < len(ALL_FEATURES):
    t = time.time()
    _, _, r_keep = quick_fit(KEEP)
    print(f"\nquick F0.5 with {len(KEEP)} features = {r_keep['f05']:.4f}   vs all {len(ALL_FEATURES)} = {r_quick['f05']:.4f}"
          f"   [{time.time()-t:.0f}s]")
    if r_keep["f05"] < r_quick["f05"] - DROP_TOL and low:
        KEEP = [f for f in ALL_FEATURES if f not in set(fixed)]
        REPORT.loc[low, "drop"] = ""
        print(f"  -> worse: gain-based drops reverted, keeping {len(KEEP)} features")
FEATURES = KEEP
REPORT["kept"] = REPORT.index.isin(FEATURES)
REPORT.index.name = "feature"
REPORT.round(6).to_csv(ART / "feature_selection.csv")
QUICK_GAIN = pd.Series(gain, index=ALL_FEATURES)
print(f"\nfeatures: {len(ALL_FEATURES)} -> {len(FEATURES)}   -> artifacts/feature_selection.csv   [{time.time()-t0:.0f}s]")
STAGE_MIN["feature selection"] = (time.time() - t0) / 60
del Xs, ys, present, P_, C, XQ; gc.collect()

empty / constant (1): ['addr_missing_s1']
near-duplicates (3):
    drop blk_n_keys                   keep blk_score                    |r| = 0.9950
    drop name_compact_lev             keep name_lev                     |r| = 0.9948
    drop s1_n_strong                  keep s1_pb_sum                    |r| = 0.9924
near-zero gain (32): ['name_set_eq', 'pin_prefix_eq', 'alias_any', 'evb_nm_tok', 'addr_exact_eq', 'evb_skF_SF', 'mutual_best_combo', 'evb_ad_full', 'f_landmark_tset', 'mutual_best_pb', 'evb_ad_skpair', 'f_subloc_eq', 'name_denoised_eq', 'name_num_eq', 'evb_nm_char', 'cand_addr_s1_cnt', 'evb_lw_skel', 'noaddr_name_eq', 'addr_street_same_num_diff', 'evb_nmF_11', 'evb_skF_11', 'evb_nmF_F1', 'fld_n_both', 'evb_s2F_FF', 'evb_nm_pfx', 'evb_skF_FF', 'name_unm_s1', 'name_overlap', 'fld_n_conflict', 'evb_nmF_SF', 'x_s1name_in_candaddr', 'cand_src']

quick F0.5 with 122 features = 0.9873   vs all 158 = 0.9874   [47s]

features: 158 -> 122   -> artifacts/feature_selection.csv   [55s]


990

## 3. Level-1 models, K-fold by S1
Each model is trained `K_FOLDS` times; fold *k*'s model never sees fold *k*'s S1s and scores them → every training pair
gets an **out-of-fold** probability (what the graph re-score in Section 4 learns from). Validation (and, in file 4, test)
gets the **mean of the K fold models**. Early stopping uses 5% of each fold's own training S1s; each model stops by the
clock (`L1_BUDGET_MIN` per model, all folds together — shrunk automatically if the 50-minute plan runs late).
CatBoost (a different tree family: symmetric trees) fits on `ROW_CAP` random rows per fold — its value is diversity:
the graph re-score sees both probabilities and their disagreement.

In [19]:
t0 = time.time()
if "FEATURES" not in globals():                       # after a restart: the saved selection
    FEATURES = pd.read_csv(ART / "feature_selection.csv").query("kept")["feature"].tolist()
    print(f"features from artifacts/feature_selection.csv: {len(FEATURES)}")
ORDER = np.concatenate([ROWS_TR, ROWS_VAL])
N_TR = len(ROWS_TR)
VAL = slice(N_TR, None)
X_ALL = load_rows(ORDER, FEATURES)
Y_ALL, S1C, CDC, CTRY = y_raw[ORDER], s1_raw[ORDER], cd_raw[ORDER], ctry_raw[ORDER]
FOLD = S1_FOLD[S1C[:N_TR]]
S1_VAL, CD_VAL, Y_VAL, CTRY_VAL = S1C[VAL], CDC[VAL], Y_ALL[VAL], CTRY[VAL]
META = dict(features=list(FEATURES), k=K_FOLDS, n=int(len(ORDER)), n_val=int(len(ROWS_VAL)), seed=SEED,
            lr=L1_LR, budgets=dict(L1_BUDGET_MIN), row_cap=dict(ROW_CAP), es=ES_FRAC)
print(f"X = {X_ALL.shape[0]:,} x {X_ALL.shape[1]}  ({X_ALL.nbytes / 1e9:.2f} GB) | train {N_TR:,} | val {len(ROWS_VAL):,}"
      f"   [{time.time()-t0:.0f}s]")

def fold_rows(k):
    """(fit, early-stop, held-out) row indices inside the training block for fold k."""
    r = np.random.default_rng(SEED + 100 + k)
    tr_c = np.flatnonzero((S1_FOLD >= 0) & (S1_FOLD != k))
    es_flag = np.zeros(len(S1_CATS), bool)
    es_flag[r.choice(tr_c, size=max(1, int(len(tr_c) * ES_FRAC)), replace=False)] = True
    e = es_flag[S1C[:N_TR]]
    return (np.flatnonzero((FOLD != k) & ~e), np.flatnonzero((FOLD != k) & e), np.flatnonzero(FOLD == k))

def cap_rows(idx, cap, k):
    if cap is None or len(idx) <= cap:
        return idx
    return np.sort(np.random.default_rng(SEED + 200 + k).choice(idx, size=cap, replace=False))

X = 13,550,327 x 122  (6.61 GB) | train 11,518,129 | val 2,032,198   [94s]


In [20]:
L1_LGB = dict(objective="binary", learning_rate=L1_LR, num_leaves=184, min_child_samples=300, subsample=0.9,
              subsample_freq=1, colsample_bytree=0.45, reg_alpha=6.0, reg_lambda=0.01, min_split_gain=0.13,
              num_threads=N_JOBS if N_JOBS > 0 else 0, seed=SEED, verbose=-1, feature_pre_filter=False)

def folds_lightgbm(X, y, params, sec, feat_names=None):
    D = lgb.Dataset(X[:N_TR], label=y[:N_TR], feature_name=feat_names or "auto",
                    params={"verbose": -1, "feature_pre_filter": False}).construct()
    models, oof = [], np.zeros(N_TR, np.float32)
    for k in range(K_FOLDS):
        fit, es, ho = fold_rows(k)
        t = time.time()
        bst = lgb.train(params, D.subset(fit), num_boost_round=50_000, valid_sets=[D.subset(es)],
                        callbacks=[lgb.early_stopping(100, verbose=False), MF.lgb_time_limit(sec)])
        m = dict(lib="lightgbm", model=bst, iters=int(bst.best_iteration or bst.current_iteration()))
        oof[ho] = MF.rows_proba(m, X, ho)
        models.append(m)
        print(f"    fold {k}: {m['iters']} trees, fit {len(fit):,} rows   [{time.time()-t:.0f}s]")
    del D; gc.collect()
    return models, oof

def folds_xgboost(X, y, sec):
    import xgboost as xgb
    prm = dict(objective="binary:logistic", eval_metric="logloss", eta=L1_LR, max_depth=9, min_child_weight=20,
               subsample=0.85, colsample_bytree=0.5, gamma=0.1, alpha=1.0, reg_lambda=2.0, tree_method="hist",
               max_bin=256, nthread=os.cpu_count(), seed=SEED)
    models, oof = [], np.zeros(N_TR, np.float32)
    for k in range(K_FOLDS):
        fit, es, ho = fold_rows(k)
        fit = cap_rows(fit, ROW_CAP.get("xgboost"), k)
        t = time.time()
        try:
            dtr = xgb.QuantileDMatrix(X[fit], label=y[fit], max_bin=256)
        except AttributeError:                                  # xgboost < 1.7
            dtr = xgb.DMatrix(X[fit], label=y[fit])
        des = xgb.DMatrix(X[es], label=y[es])
        bst = xgb.train(prm, dtr, num_boost_round=50_000, evals=[(des, "es")], early_stopping_rounds=100,
                        verbose_eval=False, callbacks=[MF.xgb_time_limit(sec)])
        del dtr, des; gc.collect()
        bi = getattr(bst, "best_iteration", None)
        m = dict(lib="xgboost", model=bst, iters=int(bi) + 1 if bi is not None else None)
        oof[ho] = MF.rows_proba(m, X, ho)
        models.append(m)
        print(f"    fold {k}: {m['iters']} trees, fit {len(fit):,} rows   [{time.time()-t:.0f}s]")
    return models, oof

def folds_catboost(X, y, sec):
    from catboost import CatBoostClassifier
    models, oof = [], np.zeros(N_TR, np.float32)
    for k in range(K_FOLDS):
        fit, es, ho = fold_rows(k)
        fit = cap_rows(fit, ROW_CAP.get("catboost"), k)
        t = time.time()
        est = CatBoostClassifier(iterations=50_000, learning_rate=0.15, depth=8, l2_leaf_reg=5.0, border_count=128,
                                 bootstrap_type="Bernoulli", subsample=0.8, loss_function="Logloss",
                                 od_type="Iter", od_wait=60, use_best_model=True,
                                 thread_count=-1, random_seed=SEED + k, verbose=0, allow_writing_files=False)
        try:
            est.fit(X[fit], y[fit], eval_set=(X[es], y[es]), callbacks=[MF.CatTimeLimit(sec)])
        except TypeError:                                       # old catboost without callbacks
            est.set_params(iterations=1500)
            est.fit(X[fit], y[fit], eval_set=(X[es], y[es]))
        m = dict(lib="catboost", model=est)
        oof[ho] = MF.rows_proba(m, X, ho)
        models.append(m)
        print(f"    fold {k}: {est.get_best_iteration()} trees, fit {len(fit):,} rows   [{time.time()-t:.0f}s]")
    return models, oof

def run_l1(name, minutes):
    bf, vf = MODEL_DIR / f"l1_{name}.joblib", MODEL_DIR / f"l1_{name}.npy"
    if not REFIT and bf.exists() and vf.exists():
        b = joblib.load(bf)
        if b.get("meta") == META:
            print(f"[{name}] reused from disk (same features, folds and settings)")
            return np.load(vf), b
    t = time.time()
    sec = minutes * 60 / K_FOLDS
    print(f"[{name}] {K_FOLDS} folds, {sec:.0f}s each   (clock: {used_min():.1f} min used, {left_min():.1f} left)")
    if name == "lightgbm":
        models, oof = folds_lightgbm(X_ALL, Y_ALL, L1_LGB, sec)
    elif name == "xgboost":
        models, oof = folds_xgboost(X_ALL, Y_ALL, sec)
    elif name == "catboost":
        models, oof = folds_catboost(X_ALL, Y_ALL, sec)
    else:
        raise ValueError(name)
    pval = np.mean([MF.model_proba(m, X_ALL[VAL]) for m in models], axis=0)
    P = np.concatenate([oof, pval]).astype(np.float32)
    b = dict(kind="kfold", name=name, models=models, features=list(FEATURES), meta=META,
             train_seconds=round(time.time() - t))
    joblib.dump(b, bf)
    np.save(vf, P)
    return P, b

L1P, L1B = {}, {}
for i, name in enumerate(L1_USED):
    plan = L1_BUDGET_MIN.get(name, 5)
    later = sum(L1_BUDGET_MIN.get(m, 5) + l1_overhead(m, L1_BUDGET_MIN.get(m, 5)) for m in L1_USED[i + 1:])
    avail = left_min() - later - FIELDS_MIN - GRAPH_ROUNDS * (ROUND_BUDGET_MIN + ROUND_OVERHEAD) - TAIL_MIN
    if name == "lightgbm":
        minutes = min(plan, max(6.0, (avail - 0.8) / 1.2))          # the main model always runs
    else:
        minutes = min(plan, avail - l1_overhead(name, plan))
        if minutes < min(1.5, plan):
            print(f"[{name}] skipped — not enough time left in the {TOTAL_BUDGET_MIN}-minute plan")
            continue
    t = time.time()
    L1P[name], L1B[name] = run_l1(name, minutes)
    STAGE_MIN[f"level-1 {name}"] = (time.time() - t) / 60
    show(name + " (fold mean)", EV_VAL.sweep(L1P[name][VAL]), time.time() - t)
    gc.collect()

[lightgbm] 3 folds, 320s each   (clock: 4.2 min used, 45.8 left)
    fold 0: 684 trees, fit 7,295,323 rows   [338s]
    fold 1: 753 trees, fit 7,292,632 rows   [374s]
    fold 2: 715 trees, fit 7,296,423 rows   [372s]
[lightgbm (fold mean)]  VALIDATION  F0.5 = 0.9886   precision = 0.9973   recall = 0.9726   (thr=0.67 one_per_cand=True)   [1163s]
[catboost] 3 folds, 100s each   (clock: 23.6 min used, 26.4 left)
    fold 0: 311 trees, fit 2,500,000 rows   [107s]
    fold 1: 309 trees, fit 2,500,000 rows   [107s]
    fold 2: 306 trees, fit 2,500,000 rows   [107s]
[catboost (fold mean)]  VALIDATION  F0.5 = 0.9865   precision = 0.9961   recall = 0.9701   (thr=0.65 one_per_cand=True)   [327s]


In [21]:
# the N_RAW highest-gain pair features for level 2 (LightGBM fold models, else the quick model)
if "lightgbm" in L1B:
    g = sum(pd.Series(m["model"].feature_importance("gain"), index=FEATURES) for m in L1B["lightgbm"]["models"])
else:
    g = QUICK_GAIN.reindex(FEATURES).fillna(0) if "QUICK_GAIN" in globals() else pd.Series(1.0, index=FEATURES)
RAW = g.sort_values(ascending=False).index[:N_RAW].tolist()
print("level-2 raw features:", RAW)
print("\nlevel-1 gain share — top 15:\n" + (g / g.sum()).sort_values(ascending=False).head(15).round(4).to_string())
X_RAW = X_ALL[:, [FEATURES.index(c) for c in RAW]].copy() if "X_ALL" in globals() else load_rows(ORDER, RAW)
if "X_ALL" in globals():
    del X_ALL
gc.collect()
print(f"X_RAW {X_RAW.shape}  — full matrix released")

level-2 raw features: ['pb_vs_other_s1', 'blk_p_block', 'nsoft_vs_other_cand', 'pb_vs_other_cand', 'cand_pb_sum', 's1_pb_sum', 'cand_n_s1', 'addr_num_overlap', 'addr_hnum_lev', 'sib_anchor_pb', 'name_unm_idf_cand', 'addr_num_jaccard', 'pb_rank_s1', 'name_idf_unshared_cand', 'combo_vs_other_cand', 'blk_cand_rank', 'addr_idf_unshared_cand', 'addr_token_set', 'lf_jacc', 'phon_softcov_cand']

level-1 gain share — top 15:
pb_vs_other_s1            0.3130
blk_p_block               0.2339
nsoft_vs_other_cand       0.1017
pb_vs_other_cand          0.0974
cand_pb_sum               0.0941
s1_pb_sum                 0.0193
cand_n_s1                 0.0153
addr_num_overlap          0.0150
addr_hnum_lev             0.0123
sib_anchor_pb             0.0121
name_unm_idf_cand         0.0062
addr_num_jaccard          0.0055
pb_rank_s1                0.0055
name_idf_unshared_cand    0.0053
combo_vs_other_cand       0.0036
X_RAW (13550327, 20)  — full matrix released


## 4. Graph re-score (level 2), K-fold, `GRAPH_ROUNDS` rounds
Where recall was lost (previous run): 74% of missed true pairs ranked 3rd+ inside their S1 (S1s with many matches),
73% had **no candidate address**, 26% were removed because the candidate's best S1 was another one. A pair model sees
one pair at a time; this level-2 model sees the **group**, built from out-of-fold probabilities over ALL pairs:

* **S1 side** — rank, lead, share of the S1's max, sum, how many likely / confident matches the S1 has
* **candidate side** — rank of this S1 among the candidate's S1s, lead, share of its best, how many S1s want it
* **confident-match similarity** — among the S1's *other* confident matches (p ≥ `ANCHOR_P`), the share with the same
  name / skeleton / first word / address / pin / house number / street as this candidate
* **duplicate S1s** (the biggest recall bucket: 14,942 of 24,628 missed pairs had a candidate whose best S1 is
  another S1) — the candidate's best *other* S1: its probability, how many likely candidates the two S1s share, the
  most candidates this S1 shares with ANY other S1, same name / skeleton / address / pin / house number, and the
  **word overlap** (Jaccard) of the two S1 names and addresses → a candidate legitimately matching two S1s
* the level-1 probabilities (+ their spread) and the `N_RAW` strongest pair features

Round 2 recomputes all group evidence from round 1's probabilities (and re-learns), so a pair can be pulled in by
matches that round 1 recovered. Each round: same S1 folds, out-of-fold on training, fold-mean on validation.

In [22]:
t0 = time.time()
SF, CF = MF.load_fields(ART, "train", S1_CATS, CD_CATS)
print("S1 fields:", {k: int((v >= 0).mean() * 1000) / 10 for k, v in SF.items() if not k.startswith("_")}, "(% filled)")
print("S1 word sets:", {k[5:]: f"{v.shape[1]:,} words" for k, v in SF.items() if k.startswith("_tok_")})
print("candidate fields:", {k: int((v >= 0).mean() * 1000) / 10 for k, v in CF.items()}, "(% filled)")
print(f"[{time.time()-t0:.0f}s]")
STAGE_MIN["graph fields"] = (time.time() - t0) / 60

L2_LGB = dict(objective="binary", learning_rate=L2_LR, num_leaves=127, min_child_samples=200, subsample=0.8,
              subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, num_threads=N_JOBS if N_JOBS > 0 else 0,
              seed=SEED, verbose=-1, feature_pre_filter=False)

def run_round(rnd, prev, minutes):
    bf, vf = MODEL_DIR / f"graph_r{rnd}.joblib", MODEL_DIR / f"graph_r{rnd}.npy"
    meta = dict(META, l1=list(L1P), rnd=rnd, raw=RAW, anchor_p=ANCHOR_P, conf_p=CONF_P, l2=dict(L2_LGB))
    if not REFIT and bf.exists() and vf.exists():
        b = joblib.load(bf)
        if b.get("meta") == meta:
            print(f"[graph_r{rnd}] reused from disk")
            return np.load(vf)
    t = time.time()
    p, ex = MF.round_inputs(rnd, L1P, prev)
    Z, names = MF.level2_matrix(p, ex, X_RAW, RAW, S1C, CDC, SF, CF, ANCHOR_P, CONF_P)
    print(f"[graph_r{rnd}] level-2 matrix {Z.shape[0]:,} x {Z.shape[1]}  ({Z.nbytes / 1e9:.2f} GB)   [{time.time()-t:.0f}s]")
    models, oof = folds_lightgbm(Z, Y_ALL, L2_LGB, minutes * 60 / K_FOLDS, feat_names=names)
    qval = np.mean([MF.model_proba(m, Z[VAL]) for m in models], axis=0)
    Q = np.concatenate([oof, qval]).astype(np.float32)
    g = sum(pd.Series(m["model"].feature_importance("gain"), index=names) for m in models)
    print("  gain share — top 15:\n" + (g / g.sum()).sort_values(ascending=False).head(15).round(4).to_string())
    joblib.dump(dict(kind="graph", round=rnd, models=models, feature_names=names, raw_features=RAW, l1=list(L1P),
                     anchor_p=ANCHOR_P, conf_p=CONF_P, meta=meta, train_seconds=round(time.time() - t)), bf)
    np.save(vf, Q)
    del Z; gc.collect()
    return Q

GRAPH = {}
prev = None
for rnd in range(1, GRAPH_ROUNDS + 1):
    after = (GRAPH_ROUNDS - rnd) * (ROUND_BUDGET_MIN + ROUND_OVERHEAD)
    minutes = min(ROUND_BUDGET_MIN, left_min() - ROUND_OVERHEAD - TAIL_MIN - after)
    if rnd > 1 and minutes < min(1.5, ROUND_BUDGET_MIN):
        print(f"[graph_r{rnd}] skipped — not enough time left in the {TOTAL_BUDGET_MIN}-minute plan")
        break
    minutes = max(minutes, min(1.5, ROUND_BUDGET_MIN))
    t = time.time()
    print(f"[graph_r{rnd}] training budget {minutes:.1f} min   (clock: {used_min():.1f} min used, {left_min():.1f} left)")
    prev = run_round(rnd, prev, minutes)
    GRAPH[rnd] = prev
    STAGE_MIN[f"graph round {rnd}"] = (time.time() - t) / 60
    show(f"graph_r{rnd}", EV_VAL.sweep(prev[VAL]), time.time() - t)

S1 fields: {'name_compact': 100.0, 'name_skeleton': 99.9, 'address_core': 99.9, 'address_pin': 1.1, 'house_number': 68.7} (% filled)
S1 word sets: {'name': '118,563 words', 'addr': '405,711 words'}
candidate fields: {'name_compact': 99.9, 'name_skeleton': 99.9, 'name_first_token': 99.9, 'address_core': 96.2, 'address_pin': 1.3, 'house_number': 62.6, 'street_name': 53.8} (% filled)
[65s]
[graph_r1] training budget 5.0 min   (clock: 30.4 min used, 19.6 left)
[graph_r1] level-2 matrix 13,550,327 x 59  (3.20 GB)   [40s]
    fold 0: 477 trees, fit 7,295,323 rows   [116s]
    fold 1: 441 trees, fit 7,292,632 rows   [112s]
    fold 2: 318 trees, fit 7,296,423 rows   [87s]
  gain share — top 15:
x_lightgbm                    0.8993
p                             0.0386
rank_minus_n50                0.0380
x_catboost                    0.0113
c_sum                         0.0010
raw_blk_p_block               0.0008
other_s1_p                    0.0007
anchor_share_house_number     0.0007
c_lead 

## 5. Decision rule, leaderboard → `best_model.json`
For every probability (each level-1 model, their mean, each graph round) the rule is tuned on validation:
1. **threshold × soft one-S1-per-candidate** — a pair survives if p ≥ `opc_r` × the candidate's best p over **all**
   its S1s (1 = only the best S1, 0.9 = also S1s almost as good — duplicate S1s; 0 = off)
2. **per-rank thresholds** — a lower bar for each S1's best / second-best surviving pair (greedy, only if F0.5 rises)
3. **per country** — the same search per country, kept only if the combined F0.5 rises (candidates never cross
   countries); a country unseen in training (France) uses the global rule

A rule with validation precision below `MIN_PRECISION` is avoided unless that costs more than `PREC_SLACK` of F0.5
(F0.5 is what is scored). The winner (highest F0.5) is written to
`best_model.json`; `4_predictions_final.py` rebuilds exactly that pipeline on test.

In [23]:
def _search(ev, q, s1, cm):
    best, best_any = None, None
    for r in OPC_GRID:
        base = q >= r * cm - 1e-12
        for t in THR_MAIN:
            res = ev.score(base & (q >= t))
            cand = dict(res, rule=dict(thr=float(t), opc_r=float(r), thr_ranks=[]))
            if best_any is None or res["f05"] > best_any["f05"]:
                best_any = cand
            if res["precision"] >= MIN_PRECISION and (best is None or res["f05"] > best["f05"]):
                best = cand
    # the precision floor may cost at most PREC_SLACK of F0.5 — F0.5 is what is scored
    best = best if best is not None and best["f05"] >= best_any["f05"] - PREC_SLACK else best_any
    rule = best["rule"]
    base = q >= rule["opc_r"] * cm - 1e-12
    pred = base & (q >= rule["thr"])
    rank, _ = MF.rank_lead(s1, np.where(base, q, -np.inf))
    for k in range(RANK_MAX):
        rows = base & (rank == k)
        bt, bf_ = None, best
        for t in THR_GRID[THR_GRID < rule["thr"]]:
            res = ev.score(pred | (rows & (q >= t)))
            if res["f05"] > bf_["f05"] + 1e-6 and res["precision"] >= min(MIN_PRECISION, best["precision"]):
                bt, bf_ = float(t), dict(res, rule=rule)
        if bt is not None:
            pred = pred | (rows & (q >= bt))
            rule = dict(rule, thr_ranks=rule["thr_ranks"] + [(k, bt)])
            best = dict(bf_, rule=rule)
    return best

VAL_S1_BY_CTRY = pd.Series(val_s1).groupby(s1_country.reindex(val_s1).fillna("").to_numpy()).apply(list).to_dict()

def tune_rule(Q):
    """Q: probability of every row (train block out-of-fold + validation) -> best rule on validation."""
    Q = np.asarray(Q, np.float64)
    cm = MF.group_max(Q, CDC)[VAL]                 # candidate's best p over ALL its S1s (train + val)
    q = Q[VAL]
    g = _search(EV_VAL, q, S1_VAL, cm)
    rule = dict(default=g["rule"], by_country={})
    best = dict(g, rule=rule)
    ctrys = [c for c in pd.unique(CTRY_VAL)]
    if len(ctrys) > 1:
        by = {}
        for c in ctrys:
            m = CTRY_VAL == c
            ev_c = F05Evaluator(VAL_S1_BY_CTRY.get(c, np.unique(S1_CATS[S1_VAL[m]])), S1_VAL[m], CD_VAL[m], Y_VAL[m])
            by[c] = _search(ev_c, q[m], S1_VAL[m], cm[m])["rule"]
        rc = dict(default=g["rule"], by_country=by)
        res = EV_VAL.score(MF.apply_rule(q, S1_VAL, CD_VAL, rc, CTRY_VAL, cmax=cm))
        if res["f05"] > best["f05"] + 1e-6 and res["precision"] >= min(MIN_PRECISION, best["precision"]):
            best = dict(res, rule=rc)
    return best

def predict_val(Q, rule):
    Q = np.asarray(Q, np.float64)
    return MF.apply_rule(Q[VAL], S1_VAL, CD_VAL, rule, CTRY_VAL, cmax=MF.group_max(Q, CDC)[VAL])

CANDIDATES = {f"l1_{m}": (dict(pipeline="level1", l1=[m], rounds=0), L1P[m]) for m in L1P}
if len(L1P) >= 2:
    CANDIDATES["l1_mean"] = (dict(pipeline="level1", l1=list(L1P), rounds=0), np.mean([L1P[m] for m in L1P], axis=0))
for rnd, Q in GRAPH.items():
    CANDIDATES[f"graph_r{rnd}"] = (dict(pipeline="graph", l1=list(L1P), rounds=rnd), Q)

T_TUNE = time.time()
LB, RULES = {}, {}
for name, (cfg, Q) in CANDIDATES.items():
    t = time.time()
    r = tune_rule(Q)
    RULES[name] = r["rule"]
    LB[name] = {k: r[k] for k in ["f05", "precision", "recall", "f05_matched", "singleton_acc", "n_pred"]}
    LB[name]["per_country"] = bool(r["rule"]["by_country"])
    rd = r["rule"]["default"]
    print(f"[{name:<14}] F0.5 {r['f05']:.4f}  P {r['precision']:.4f}  R {r['recall']:.4f}   "
          f"thr {rd['thr']:.2f}  opc_r {rd['opc_r']}  ranks {rd['thr_ranks']}"
          f"{'  + per-country rules' if r['rule']['by_country'] else ''}   [{time.time()-t:.0f}s]")

LBD = pd.DataFrame(LB).T.sort_values("f05", ascending=False)
print(f"\nVALIDATION LEADERBOARD  (oracle on these candidates = {oracle['f05']:.4f})\n")
print(LBD.to_string(float_format=lambda v: f"{v:.4f}"))

BEST = LBD.index[0]
cfg = dict(CANDIDATES[BEST][0], name=BEST, features=list(FEATURES), rule=RULES[BEST],
           val={k: float(LBD.loc[BEST, k]) for k in ["f05", "precision", "recall"]},
           k_folds=K_FOLDS, raw=RAW, anchor_p=ANCHOR_P, conf_p=CONF_P)
json.dump(cfg, open(MODEL_DIR / "best_model.json", "w"), indent=2, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
print(f"\nbest: {BEST}  -> artifacts/models/best_model.json   (4_predictions_final.py applies it)")

[l1_lightgbm   ] F0.5 0.9888  P 0.9972  R 0.9728   thr 0.66  opc_r 1.0  ranks [(0, 0.49)]  + per-country rules   [21s]
[l1_catboost   ] F0.5 0.9867  P 0.9966  R 0.9690   thr 0.65  opc_r 1.0  ranks [(0, 0.47)]  + per-country rules   [19s]
[l1_mean       ] F0.5 0.9881  P 0.9968  R 0.9722   thr 0.63  opc_r 1.0  ranks [(0, 0.58)]  + per-country rules   [18s]
[graph_r1      ] F0.5 0.9897  P 0.9981  R 0.9736   thr 0.70  opc_r 0.97  ranks [(0, 0.52), (1, 0.69)]   [19s]
[graph_r2      ] F0.5 0.9897  P 0.9978  R 0.9745   thr 0.65  opc_r 1.0  ranks [(0, 0.52)]   [17s]

VALIDATION LEADERBOARD  (oracle on these candidates = 0.9985)

               f05 precision recall f05_matched singleton_acc   n_pred per_country
graph_r1    0.9897    0.9981 0.9736      0.9896        0.9907  1116388       False
graph_r2    0.9897    0.9978 0.9745      0.9896        0.9906  1117787       False
l1_lightgbm 0.9888    0.9972 0.9728      0.9889        0.9867  1116431        True
l1_mean     0.9881    0.9968 0.9722    

In [24]:
# ---------- where the F0.5 is lost: first level-1 model vs the winner ----------
first = f"l1_{list(L1P)[0]}"
for nm in dict.fromkeys([first, BEST]):
    EV_VAL.loss_report(predict_val(CANDIDATES[nm][1], RULES[nm]), f"— {nm}")

# missed true pairs of the winner
Qb = np.asarray(CANDIDATES[BEST][1], np.float64)
pred_b = predict_val(Qb, RULES[BEST])
qb = Qb[VAL]
fn = (Y_VAL == 1) & ~pred_b
cm_b = MF.group_max(Qb, CDC)[VAL]
rk, _ = MF.rank_lead(S1_VAL, qb)
print(f"\n{BEST}: missed true pairs {int(fn.sum()):,} of {int(Y_VAL.sum()):,} (in candidates),"
      f"  wrong pairs {int((pred_b & (Y_VAL == 0)).sum()):,}")
print("  by probability band :", pd.cut(qb[fn], [0, .05, .2, .4, .6, .8, 1.0], include_lowest=True).value_counts().sort_index().to_dict())
print(f"  candidate's best S1 is another S1 : {int((fn & (qb < cm_b - 1e-12)).sum()):,}")
print("  rank inside the S1 :", pd.Series(np.minimum(rk[fn], 5)).value_counts().sort_index().to_dict())

# the duplicate-S1 bucket: is the candidate's preferred (other) S1 pair ALSO a true match?
other_all = MF._top2_other(CDC, Qb, S1C.astype(np.int64))
o_val = other_all[VAL]
m_opc = fn & (qb < cm_b - 1e-12) & (o_val >= 0)
if m_opc.any():
    ncd = int(CDC.max()) + 1
    key_all = S1C.astype(np.int64) * ncd + CDC
    order = np.argsort(key_all, kind="stable")
    ks, ys = key_all[order], Y_ALL[order]
    qk = o_val[m_opc].astype(np.int64) * ncd + CD_VAL[m_opc]
    pos = np.minimum(np.searchsorted(ks, qk), len(ks) - 1)
    both = (ks[pos] == qk) & (ys[pos] == 1)
    print(f"  of the {int(m_opc.sum()):,} missed pairs whose candidate prefers another S1: that other pair is ALSO "
          f"a true match in {both.mean():.1%} (duplicate S1s)")
STAGE_MIN["rule tuning + reports"] = (time.time() - T_TUNE) / 60
print("\nTIME (minutes):", {k: round(v, 1) for k, v in STAGE_MIN.items()},
      f"| total {used_min():.1f} of {TOTAL_BUDGET_MIN}")


F0.5 points lost — l1_lightgbm  (F0.5 = 1 - ALL.total lost)
kind              S1s  blocking miss only  missed pair(s)  singleton predicted  wrong + missed  wrong pair(s)  total lost
bucket                                                                                                                   
0              15,533             0.00000         0.00000              0.00076         0.00000        0.00000     0.00076
1              17,838             0.00022         0.00143              0.00000         0.00002        0.00030     0.00198
2              56,258             0.00029         0.00132              0.00000         0.00011        0.00048     0.00220
3-5           200,369             0.00073         0.00360              0.00000         0.00021        0.00080     0.00534
6-10           37,804             0.00011         0.00067              0.00000         0.00003        0.00007     0.00088
no candidates   3,221             0.00004         0.00000              0.00000       